In [ ]:
# ============================================================================
# SECTION 11: SVD++ MODEL TRAINING (Using Surprise)
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 11: SVD++ MODEL TRAINING")
print("=" * 60)

from surprise import SVDpp, Dataset, Reader
from surprise.model_selection import train_test_split, cross_validate
import pandas as pd

# Load the prepared data
interactions = pd.read_csv('../data/processed/interactions_clean.csv')
print(f"Loaded {len(interactions):,} interactions")

# Prepare for Surprise
reader = Reader(rating_scale=(-3.0, 3.0))  # Double-centered range
data = Dataset.load_from_df(
    interactions[['userId', 'tmdbId', 'explicit_value']], 
    reader
)

# Train SVD++
print("\nTraining SVD++ model...")
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

model = SVDpp(
    n_factors=100,
    n_epochs=30,
    lr_all=0.007,
    reg_all=0.02,
    random_state=42
)

model.fit(trainset)

# Evaluate
from surprise import accuracy
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
print(f"RMSE: {rmse:.4f}")

# Save model
import pickle
with open('../models/cf/svdpp_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("✓ Saved: svdpp_model.pkl")